# 🤪 Variational Autoencoders - CelebA Faces

In this notebook, we'll walk through the steps required to train your own variational autoencoder on the CelebA faces dataset

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from torchvision.utils import save_image

from scipy.stats import norm
import pandas as pd

import sys
from pathlib import Path

# Make the repo root importable so `notebooks.utils_pytorch` can be found
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils_pytorch import sample_batch, display

from vae_utils_pytorch import get_vector_from_label, add_vector_to_images, morph_faces

# Check GPU - PyTorch needs to be told which device to use
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():  # Apple Silicon
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 32
CHANNELS = 3
BATCH_SIZE = 128
NUM_FEATURES = 128
Z_DIM = 200
LEARNING_RATE = 0.0005
EPOCHS = 10
BETA = 2000
LOAD_MODEL = False

## 1. Prepare the data <a name="prepare"></a>

In [ ]:
# Load the data
class CelebADataset(Dataset):
    """All the .jpg images found (recursively) in a folder, in sorted order,
    with an optional list of labels - like Keras' image_dataset_from_directory."""

    def __init__(self, folder, labels=None, transform=None):
        self.files = sorted(Path(folder).rglob("*.jpg"))
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.labels is None:
            return img
        return img, self.labels[idx]


train_data = CelebADataset(
    "../../../data/celeba-dataset/img_align_celeba/img_align_celeba",
    transform=transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE), interpolation=transforms.InterpolationMode.BILINEAR
    ),
)
print(f"Found {len(train_data)} files.")

In [ ]:
# Preprocess the data
# ToTensor turns a PIL image (H, W, C) with values 0-255 into a
# float tensor (C, H, W) with values in [0, 1]
preprocess = transforms.Compose([train_data.transform, transforms.ToTensor()])
train_data.transform = preprocess

# The DataLoader batches and shuffles the data (raise num_workers to load faster)
train = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
train_sample = sample_batch(train)

In [ ]:
# Show some faces from the training set
display(train_sample, cmap=None)

## 2. Build the variational autoencoder <a name="build"></a>

In [ ]:
class Sampling(nn.Module):
    """The reparameterization trick: z = mean + std * epsilon, epsilon ~ N(0, I)."""

    def forward(self, z_mean, z_log_var):
        epsilon = torch.randn_like(z_mean)
        return z_mean + torch.exp(0.5 * z_log_var) * epsilon

In [ ]:
# Encoder
# Each block is Conv -> BatchNorm -> LeakyReLU. Differences from Keras defaults:
#  - padding=1 with a 3x3 kernel plays the role of padding="same"
#  - BatchNorm momentum: Keras momentum=0.99 is PyTorch momentum=0.01
#  - Keras' LeakyReLU uses a slope of 0.3 (PyTorch's default is 0.01)
shape_before_flattening = (NUM_FEATURES, IMAGE_SIZE // 16, IMAGE_SIZE // 16)  # the decoder will need this!


def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1),
        nn.BatchNorm2d(out_channels, momentum=0.01),
        nn.LeakyReLU(0.3),
    )


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            conv_block(CHANNELS, NUM_FEATURES),  # 32 -> 16
            conv_block(NUM_FEATURES, NUM_FEATURES),  # 16 -> 8
            conv_block(NUM_FEATURES, NUM_FEATURES),  # 8 -> 4
            conv_block(NUM_FEATURES, NUM_FEATURES),  # 4 -> 2
            nn.Flatten(),
        )
        self.z_mean = nn.Linear(int(np.prod(shape_before_flattening)), Z_DIM)
        self.z_log_var = nn.Linear(int(np.prod(shape_before_flattening)), Z_DIM)
        self.sampling = Sampling()

    def forward(self, x):
        x = self.conv(x)
        z_mean = self.z_mean(x)
        z_log_var = self.z_log_var(x)
        z = self.sampling(z_mean, z_log_var)
        return z_mean, z_log_var, z


encoder = Encoder().to(device)
print(encoder)
print(f"Total params: {sum(p.numel() for p in encoder.parameters()):,}")

In [ ]:
# Decoder
def deconv_block(in_channels, out_channels):
    # stride=2, padding=1, output_padding=1 doubles the size exactly,
    # like Keras' Conv2DTranspose(strides=2, padding="same")
    return nn.Sequential(
        nn.ConvTranspose2d(
            in_channels, out_channels, kernel_size=3, stride=2, padding=1, output_padding=1
        ),
        nn.BatchNorm2d(out_channels, momentum=0.01),
        nn.LeakyReLU(0.3),
    )


class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(Z_DIM, int(np.prod(shape_before_flattening))),
            nn.BatchNorm1d(int(np.prod(shape_before_flattening)), momentum=0.01),
            nn.LeakyReLU(0.3),
        )
        self.deconv = nn.Sequential(
            deconv_block(NUM_FEATURES, NUM_FEATURES),  # 2 -> 4
            deconv_block(NUM_FEATURES, NUM_FEATURES),  # 4 -> 8
            deconv_block(NUM_FEATURES, NUM_FEATURES),  # 8 -> 16
            deconv_block(NUM_FEATURES, NUM_FEATURES),  # 16 -> 32
            nn.ConvTranspose2d(NUM_FEATURES, CHANNELS, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid(),  # pixel values in [0, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, *shape_before_flattening)  # Reshape
        return self.deconv(x)


decoder = Decoder().to(device)
print(decoder)
print(f"Total params: {sum(p.numel() for p in decoder.parameters()):,}")

In [ ]:
class VAE(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, inputs):
        """Call the model on a particular input."""
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)
        return z_mean, z_log_var, reconstruction

    def compute_losses(self, data):
        """Forward pass + the three VAE losses."""
        z_mean, z_log_var, reconstruction = self(data)
        # Mean squared error over all pixels (and the batch), scaled by BETA
        reconstruction_loss = BETA * F.mse_loss(reconstruction, data)
        # KL divergence to N(0, I): summed over latent dims, averaged over the batch
        kl_loss = torch.mean(
            torch.sum(-0.5 * (1 + z_log_var - z_mean.pow(2) - z_log_var.exp()), dim=1)
        )
        total_loss = reconstruction_loss + kl_loss
        return {
            "loss": total_loss,
            "reconstruction_loss": reconstruction_loss,
            "kl_loss": kl_loss,
        }

In [ ]:
# Create a variational autoencoder
vae = VAE(encoder, decoder).to(device)

## 3. Train the variational autoencoder <a name="train"></a>

In [ ]:
# "Compile": in PyTorch we simply create the optimizer ourselves
optimizer = torch.optim.Adam(vae.parameters(), lr=LEARNING_RATE)

In [ ]:
# Checkpointing and TensorBoard logging are done by hand in the training loop below:
# keep the weights with the lowest training loss, and write losses to ./logs
best_loss = float("inf")
writer = SummaryWriter(log_dir="./logs")


def generate_images(decoder, num_img, latent_dim, epoch):
    """Decode random latent vectors and save them to ./output (run after each epoch)."""
    decoder.eval()  # BatchNorm in inference mode
    with torch.no_grad():
        random_latent_vectors = torch.randn(num_img, latent_dim, device=device)
        generated_images = decoder(random_latent_vectors)
    for i in range(num_img):
        save_image(generated_images[i], "./output/generated_img_%03d_%d.png" % (epoch, i))

In [ ]:
# Load old weights if required
if LOAD_MODEL:
    vae.load_state_dict(torch.load("./models/vae.pt", map_location=device))

In [ ]:
for epoch in range(EPOCHS):
    vae.train()  # BatchNorm uses batch statistics while training
    sums = {"loss": 0.0, "reconstruction_loss": 0.0, "kl_loss": 0.0}
    n_seen = 0
    for data in train:
        data = data.to(device)
        losses = vae.compute_losses(data)

        optimizer.zero_grad()  # reset gradients from the previous step
        losses["loss"].backward()  # backpropagation
        optimizer.step()  # update the weights

        for k in sums:
            sums[k] += losses[k].item() * len(data)
        n_seen += len(data)

    logs = {k: v / n_seen for k, v in sums.items()}
    print(f"Epoch {epoch + 1}/{EPOCHS} - " + " - ".join(f"{k}: {v:.4f}" for k, v in logs.items()))
    for k, v in logs.items():
        writer.add_scalar(k, v, epoch)

    # Model checkpoint: save only when the training loss improves
    if logs["loss"] < best_loss:
        best_loss = logs["loss"]
        torch.save(vae.state_dict(), "./checkpoint/vae.pt")

    # Image generator "callback"
    generate_images(decoder, num_img=10, latent_dim=Z_DIM, epoch=epoch)

writer.close()

In [ ]:
# Save the final models (PyTorch saves the weights, i.e. the state_dict)
torch.save(vae.state_dict(), "./models/vae.pt")
torch.save(encoder.state_dict(), "./models/encoder.pt")
torch.save(decoder.state_dict(), "./models/decoder.pt")

## 3. Reconstruct using the variational autoencoder <a name="reconstruct"></a>

In [ ]:
# Select a subset of the test set
batches_to_predict = 1
example_images = torch.cat([batch for _, batch in zip(range(batches_to_predict), train)])

In [ ]:
# Create autoencoder predictions and display
vae.eval()  # inference mode (BatchNorm uses its running statistics)
with torch.no_grad():
    z_mean, z_log_var, reconstructions = vae(example_images.to(device))
print("Example real faces")
display(example_images)
print("Reconstructions")
display(reconstructions)

## 4. Latent space distribution

In [ ]:
with torch.no_grad():
    _, _, z = vae.encoder(example_images.to(device))
z = z.cpu().numpy()

x = np.linspace(-3, 3, 100)

fig = plt.figure(figsize=(20, 5))
fig.subplots_adjust(hspace=0.6, wspace=0.4)

for i in range(50):
    ax = fig.add_subplot(5, 10, i + 1)
    ax.hist(z[:, i], density=True, bins=20)
    ax.axis("off")
    ax.text(
        0.5, -0.35, str(i), fontsize=10, ha="center", transform=ax.transAxes
    )
    ax.plot(x, norm.pdf(x))

plt.show()

## 5. Generate new faces <a name="decode"></a>

In [ ]:
# Sample some points in the latent space, from the standard normal distribution
grid_width, grid_height = (10, 3)
z_sample = np.random.normal(size=(grid_width * grid_height, Z_DIM))

In [ ]:
# Decode the sampled points
with torch.no_grad():
    reconstructions = decoder(torch.tensor(z_sample, dtype=torch.float32, device=device))
reconstructions = reconstructions.permute(0, 2, 3, 1).cpu().numpy()  # NCHW -> NHWC for imshow

In [ ]:
# Draw a plot of decoded images
fig = plt.figure(figsize=(18, 5))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

# Output the grid of faces
for i in range(grid_width * grid_height):
    ax = fig.add_subplot(grid_height, grid_width, i + 1)
    ax.axis("off")
    ax.imshow(reconstructions[i, :, :])

## 6. Manipulate the images <a name="manipulate"></a>

In [ ]:
# Load the label dataset
attributes = pd.read_csv("../../../data/celeba-dataset/list_attr_celeba.csv")
print(attributes.columns)
attributes.head()

In [ ]:
# Load the face data with label attached
LABEL = "Blond_Hair"  # <- Set this label
labelled_data = CelebADataset(
    "../../../data/celeba-dataset/img_align_celeba",
    labels=attributes[LABEL].tolist(),  # +1 / -1, in the same (sorted) file order
    transform=preprocess,
)

# Hold out 20% of the images (like validation_split=0.2, subset="validation")
_, labelled_test = random_split(
    labelled_data, [0.8, 0.2], generator=torch.Generator().manual_seed(42)
)

labelled = DataLoader(labelled_test, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# Find the attribute vector
attribute_vec = get_vector_from_label(labelled, vae, Z_DIM, LABEL)

In [ ]:
# Add vector to images
add_vector_to_images(labelled, vae, attribute_vec)

In [ ]:
morph_faces(labelled, vae)